# Week 2 Task — Exploratory Data Analysis and Visualization
**Dataset:** Online Retail II — UK-based e-commerce transactions (Dec 2009 – Dec 2011)
**Input:** `online_retail_cleaned.csv` (output of the Week 1 cleaning task)

This notebook loads the cleaned dataset, computes summary statistics, and produces
the visualizations discussed in the Week 2 report.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/online_retail_cleaned.csv"  # adjust path as needed

df = pd.read_csv(DATA_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df.head()

In [ ]:
df.info()

## 2. Dataset Overview

Basic shape, missingness, and high-level counts.

In [ ]:
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
print(f"Unique invoices:  {df['Invoice'].nunique():,}")
print(f"Unique products:  {df['StockCode'].nunique():,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Countries:        {df['Country'].nunique()}")
print(f"Date range:       {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")
print()
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
df.describe()

## 3. Prepare Sales vs. Cancellations

`IsCancellation` flags returned/cancelled orders. These are excluded from
revenue and product analyses (they'd otherwise distort totals with negative
quantities), but kept for the cancellation-rate analysis in Section 10.

In [ ]:
sales = df[~df["IsCancellation"]].copy()

cancel_rate = df["IsCancellation"].mean()
print(f"Cancellation rate: {cancel_rate:.2%}")
print(f"Total revenue (non-cancelled): £{sales['TotalAmount'].sum():,.2f}")

## 4. Monthly Revenue Trend

In [ ]:
monthly = sales.groupby(sales["InvoiceDate"].dt.to_period("M"))["TotalAmount"].sum()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly.index, monthly.values, marker="o", color="#2E86AB")
ax.set_title("Monthly Revenue Trend (Dec 2009 - Dec 2011)", fontsize=13, fontweight="bold")
ax.set_xlabel("Month"); ax.set_ylabel("Revenue (£)")
plt.xticks(rotation=75, fontsize=8)
plt.tight_layout()
plt.show()

**Observation:** Revenue peaks sharply every November ahead of the Christmas
shopping season, consistent with the retailer's gift-focused catalog.

## 5. Revenue by Country

In [ ]:
top_countries = sales.groupby("Country")["TotalAmount"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index,
            palette="viridis", legend=False, ax=ax)
ax.set_title("Top 10 Countries by Revenue", fontsize=13, fontweight="bold")
ax.set_xlabel("Revenue (£)"); ax.set_ylabel("")
plt.tight_layout()
plt.show()

top_countries

## 6. Best-Selling Products

In [ ]:
top_products = sales.groupby("Description")["TotalAmount"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=top_products.values, y=[d[:35] for d in top_products.index],
            hue=top_products.index, palette="mako", legend=False, ax=ax)
ax.set_title("Top 10 Products by Revenue", fontsize=13, fontweight="bold")
ax.set_xlabel("Revenue (£)"); ax.set_ylabel("")
plt.tight_layout()
plt.show()

top_products

## 7. Distribution of Invoice Value

Invoice totals are heavily right-skewed, so we view them on a log10 scale.

In [ ]:
invoice_value = sales.groupby("Invoice")["TotalAmount"].sum()
invoice_value = invoice_value[invoice_value > 0]

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(np.log10(invoice_value), bins=50, color="#A23B72", ax=ax)
ax.set_title("Distribution of Invoice Value (log10 scale)", fontsize=13, fontweight="bold")
ax.set_xlabel("log10(Invoice Total £)"); ax.set_ylabel("Number of Invoices")
plt.tight_layout()
plt.show()

print(invoice_value.describe())

## 8. Purchasing Rhythm — Day of Week vs. Hour of Day

In [ ]:
sales["DayOfWeek"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = sales.pivot_table(index="DayOfWeek", columns="Hour", values="TotalAmount",
                           aggfunc="sum", fill_value=0).reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Revenue (£)"})
ax.set_title("Revenue Heatmap: Day of Week vs Hour of Day", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Customer Spend Distribution

Total spend per customer, excluding rows with a missing `Customer ID`.

In [ ]:
cust_spend = sales.dropna(subset=["Customer ID"]).groupby("Customer ID")["TotalAmount"].sum()

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(np.log10(cust_spend[cust_spend > 0]), bins=50, color="#F18F01", ax=ax)
ax.set_title("Distribution of Total Customer Spend (log10 scale)", fontsize=13, fontweight="bold")
ax.set_xlabel("log10(Total Spend £)"); ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.show()

print(cust_spend.describe())

## 10. Cancellation Rate by Country

Restricted to the 10 countries with the highest order volume, so rates
aren't noisy estimates from a handful of orders.

In [ ]:
top_vol_countries = df["Country"].value_counts().head(10).index
cancel_by_country = (
    df[df["Country"].isin(top_vol_countries)]
    .groupby("Country")["IsCancellation"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=cancel_by_country.values * 100, y=cancel_by_country.index,
            hue=cancel_by_country.index, palette="rocket", legend=False, ax=ax)
ax.set_title("Cancellation Rate (%) by Country (Top 10 by Order Volume)", fontsize=13, fontweight="bold")
ax.set_xlabel("Cancellation Rate (%)"); ax.set_ylabel("")
plt.tight_layout()
plt.show()

cancel_by_country

## 11. Summary of Findings

- Revenue is strongly seasonal, peaking every **November** ahead of Christmas.
- **United Kingdom** accounts for ~85% of revenue; EIRE, Netherlands, and Germany form a
  distant second tier.
- Best sellers are decorative/gifting items (cakestands, t-light holders, bunting) rather
  than utilitarian goods.
- Invoice value and customer spend are both **right-skewed / roughly log-normal** —
  a small number of high-value orders/customers drive a disproportionate share of revenue.
- Purchasing activity concentrates on **weekdays, 10:00-15:00**, with a trough on Saturdays.
- Overall cancellation rate is low (**1.86%**) but varies by country.

These patterns — especially the skewed customer-spend distribution — motivate the
customer-segmentation (clustering) analysis in the Week 3 task.